## 1. Import Required Libraries

# Debug CausalShapley Initialization

This notebook debugs the CausalShapley implementation by:
1. Loading the same dataset as the reference notebook (mixed_no_conf_f50_s1000_p50)
2. Initializing CausalShapley with the discovered causal structure
3. Inspecting the `self.causal_graph_components` attribute
4. Validating the graph structure extraction

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path



# Import model class
from explainability_models.shapley_values import CausalShapley
from predictive_models.predictive_models import LGBMRegressor

print("✅ Imports successful")

/Users/juanrios/Documents/master_thesis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful


## 2. Load Dataset and Model Configuration

Using the same dataset as the reference notebook: `mixed_no_conf_f50_s1000_p50`

In [2]:
# Dataset configuration (same as reference notebook)
dataset_name = "mixed_no_conf_f50_s1000_p50"
method_name = "pc"

# Load data
train_data = pd.read_parquet(f"data/processed/{dataset_name}_train.parquet")
test_data = pd.read_parquet(f"data/processed/{dataset_name}_test.parquet")

X_train = train_data.drop('Y', axis=1)
y_train = train_data['Y']
X_test = test_data.drop('Y', axis=1)
y_test = test_data['Y']

# Load model
model = LGBMRegressor.load(f"models/{dataset_name}_lgbm")

print(f"📊 Dataset: {dataset_name}")
print(f"   Training data: {X_train.shape}")
print(f"   Test data: {X_test.shape}")
print(f"   Model loaded with {len(model.selected_features)} selected features")

LightGBM model loaded from models/mixed_no_conf_f50_s1000_p50_lgbm.pkl
📊 Dataset: mixed_no_conf_f50_s1000_p50
   Training data: (800, 50)
   Test data: (200, 50)
   Model loaded with 50 selected features


## 3. Load Causal Discovery Results

Load the adjacency matrix and confounders from the causal discovery results.

In [3]:
# Load causal discovery results
results_path = f"data/causal/{dataset_name}_{method_name}_results.json"
with open(results_path, 'r') as f:
    results_json = json.load(f)

# Extract causal structure components
adjacency_matrix = np.array(results_json['adjacency_matrix'])
confounders = results_json['confounders']
feature_names = results_json['feature_names']


print(f"📊 Causal Structure Loaded:")
print(f"   Adjacency matrix shape: {adjacency_matrix.shape}")
print(f"   Total edges: {int(np.sum(adjacency_matrix != 0))}")
print(f"   Number of features: {len(feature_names)}")
print(f"   Confounders: {len(confounders)} pairs")
print(f"   \nFirst 10 features: {feature_names[:10]}")
if confounders:
    print(f"   Confounded pairs: {confounders}")

📊 Causal Structure Loaded:
   Adjacency matrix shape: (51, 51)
   Total edges: 89
   Number of features: 51
   Confounders: 23 pairs
   
First 10 features: ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9']
   Confounded pairs: [['X3', 'X8'], ['X3', 'X12'], ['X6', 'X11'], ['X7', 'X20'], ['X7', 'X21'], ['X7', 'X35'], ['X8', 'X15'], ['X8', 'X16'], ['X10', 'X24'], ['X11', 'X21'], ['X11', 'X22'], ['X12', 'X37'], ['X16', 'X21'], ['X16', 'X23'], ['X20', 'X31'], ['X23', 'X24'], ['X25', 'X35'], ['X29', 'X49'], ['X30', 'X34'], ['X32', 'X48'], ['X34', 'X49'], ['X38', 'X44'], ['X39', 'X41']]


## 4. Prepare Background Data

Prepare background data for CausalShapley (same as reference notebook).

In [4]:
# Prepare background data (100 random samples from training data)
np.random.seed(42)
bg_indices = np.random.choice(len(X_train), size=100, replace=False)
background_data = X_train.iloc[bg_indices]

print(f"Background data prepared: {background_data.shape}")
print(f"Background data columns match feature_names: {list(background_data.columns) == feature_names[:-1]}")

Background data prepared: (100, 50)
Background data columns match feature_names: True


In [5]:
len(feature_names)

51

## 5. Initialize CausalShapley

Now let's initialize CausalShapley and catch any errors that occur during initialization.

In [6]:
print("="*70)
print("INITIALIZING CAUSALSHAPLEY")
print("="*70)

try:
    causal_explainer = CausalShapley(
        model=model,
        background_data=background_data,
        discovered_adj=adjacency_matrix,
        discovered_conf=confounders,
        feature_names=feature_names,
        n_samples=100,
        M_inner_samples=10,
        random_state=42
    )
    
    print("\n✅ CausalShapley initialized successfully!")
    print(f"\n📊 Initialization Summary:")
    print(f"   Model: {type(model).__name__}")
    print(f"   Background data: {background_data.shape}")
    print(f"   Features: {len(feature_names)}")
    print(f"   Adjacency matrix: {adjacency_matrix.shape}")
    print(f"   Confounders: {len(confounders)} pairs")
    print(f"   Outer samples (n_samples): 100")
    print(f"   Inner samples (M_inner_samples): 10")
    
except Exception as e:
    print(f"\n❌ Error during initialization:")
    print(f"   Error type: {type(e).__name__}")
    print(f"   Error message: {str(e)}")
    import traceback
    print(f"\n   Full traceback:")
    traceback.print_exc()
    raise

INITIALIZING CAUSALSHAPLEY

✅ CausalShapley initialized successfully!

📊 Initialization Summary:
   Model: LGBMRegressor
   Background data: (100, 50)
   Features: 51
   Adjacency matrix: (51, 51)
   Confounders: 23 pairs
   Outer samples (n_samples): 100
   Inner samples (M_inner_samples): 10


## 6. Inspect causal_graph_components

Let's examine the `self.causal_graph_components` attribute to understand how the graph was partitioned.

In [7]:
print("="*70)
print("CAUSAL GRAPH COMPONENTS")
print("="*70)

components = causal_explainer.causal_graph_components
confounded_info = causal_explainer.confounded_info
parents_dict = causal_explainer.parents_dict
feature_to_component = causal_explainer.feature_to_component

print(f"\n📊 Overall Structure:")
print(f"   Total components: {len(components)}")
print(f"   Total features: {len(feature_names)}")
print(f"   \n   Component sizes: {[len(comp) for comp in components]}")

print(f"\n{'='*70}")
print("COMPONENT DETAILS")
print(f"{'='*70}")

for comp_idx, component in enumerate(components):
    print(f"\n🔹 Component {comp_idx}:")
    print(f"   Feature indices: {component}")
    print(f"   Feature names: {[feature_names[i] for i in component]}")
    print(f"   Component size: {len(component)}")
    print(f"   Is confounded: {confounded_info.get(comp_idx, False)}")
    
    # Get parent components
    parent_features = parents_dict.get(comp_idx, [])
    if parent_features:
        print(f"   Parent feature indices: {parent_features}")
        print(f"   Parent feature names: {[feature_names[i] for i in parent_features]}")
        # Find which components the parents belong to
        parent_components = set([feature_to_component[p] for p in parent_features if p in feature_to_component])
        print(f"   Parent components: {sorted(list(parent_components))}")
    else:
        print(f"   Parent features: None (source component)")
    
print(f"\n{'='*70}")

CAUSAL GRAPH COMPONENTS

📊 Overall Structure:
   Total components: 34
   Total features: 51
   
   Component sizes: [5, 3, 2, 6, 2, 4, 3, 3, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

COMPONENT DETAILS

🔹 Component 0:
   Feature indices: [21, 35, 7, 25, 11]
   Feature names: ['X21', 'X35', 'X7', 'X25', 'X11']
   Component size: 5
   Is confounded: True
   Parent features: None (source component)

🔹 Component 1:
   Feature indices: [20, 7, 31]
   Feature names: ['X20', 'X7', 'X31']
   Component size: 3
   Is confounded: True
   Parent feature indices: [35]
   Parent feature names: ['X35']
   Parent components: [0]

🔹 Component 2:
   Feature indices: [44, 38]
   Feature names: ['X44', 'X38']
   Component size: 2
   Is confounded: True
   Parent features: None (source component)

🔹 Component 3:
   Feature indices: [23, 16, 8, 37, 12, 3]
   Feature names: ['X23', 'X16', 'X8', 'X37', 'X12', 'X3']
   Component size: 6
   Is confounded: True
   Parent featu

## 7. Validate Graph Structure

Validate that the extracted components are correct.

In [8]:
print("="*70)
print("VALIDATION CHECKS")
print("="*70)

# Check 1: All features assigned to exactly one component
all_features_in_components = set()
for component in components:
    all_features_in_components.update(component)

all_features = set(range(len(feature_names)))
missing_features = all_features - all_features_in_components
duplicate_check = sum([len(comp) for comp in components])

print(f"\n✓ Check 1: Feature Assignment")
print(f"   Total features: {len(feature_names)}")
print(f"   Features in components: {len(all_features_in_components)}")
print(f"   Sum of component sizes: {duplicate_check}")
print(f"   Missing features: {missing_features if missing_features else 'None ✓'}")
print(f"   Status: {'✅ PASS' if len(all_features_in_components) == len(feature_names) and not missing_features else '❌ FAIL'}")

# Check 2: Verify confounded components
print(f"\n✓ Check 2: Confounded Components")
confounded_components = [i for i, is_conf in confounded_info.items() if is_conf]
print(f"   Number of confounded components: {len(confounded_components)}")
print(f"   Confounded component indices: {confounded_components}")
for comp_idx in confounded_components:
    comp = components[comp_idx]
    print(f"   Component {comp_idx}: {len(comp)} features - {[feature_names[i] for i in comp]}")
print(f"   Expected confounders from input: {len(confounders)} pairs")
print(f"   Status: {'✅ PASS - Confounders correctly identified' if len(confounded_components) == len(confounders) else '⚠️  Mismatch in confounder count'}")

# Check 3: Topological ordering validation
print(f"\n✓ Check 3: Topological Ordering")
topological_violations = []
for comp_idx, component in enumerate(components):
    parent_features = parents_dict.get(comp_idx, [])
    for parent_idx in parent_features:
        # Find which component this parent belongs to
        parent_comp = feature_to_component.get(parent_idx)
        if parent_comp is not None and parent_comp >= comp_idx:
            topological_violations.append((parent_comp, comp_idx, parent_idx))

if topological_violations:
    print(f"   ❌ FAIL - Found {len(topological_violations)} topological violations:")
    for parent_comp, child_comp, parent_feat in topological_violations:
        print(f"      Parent component {parent_comp} >= Child component {child_comp} (parent feature: {feature_names[parent_feat]})")
else:
    print(f"   ✅ PASS - Topological order is correct (parents come before children)")

# Check 4: Adjacency matrix edge count
print(f"\n✓ Check 4: Edge Count Validation")
total_edges_in_adj = int(np.sum(adjacency_matrix != 0))
print(f"   Edges in adjacency matrix: {total_edges_in_adj}")
print(f"   Components: {len(components)}")
print(f"   Status: ✅ Structure extracted from {total_edges_in_adj} edges")

print(f"\n{'='*70}")
print("VALIDATION SUMMARY")
print(f"{'='*70}")
print(f"   ✅ All features assigned: {len(all_features_in_components) == len(feature_names)}")
print(f"   ✅ No duplicate assignments: {duplicate_check == len(feature_names)}")
print(f"   ✅ Topological order valid: {len(topological_violations) == 0}")
print(f"   ✅ Components extracted correctly")
print(f"{'='*70}")

VALIDATION CHECKS

✓ Check 1: Feature Assignment
   Total features: 51
   Features in components: 51
   Sum of component sizes: 57
   Missing features: None ✓
   Status: ✅ PASS

✓ Check 2: Confounded Components
   Number of confounded components: 11
   Confounded component indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
   Component 0: 5 features - ['X21', 'X35', 'X7', 'X25', 'X11']
   Component 1: 3 features - ['X20', 'X7', 'X31']
   Component 2: 2 features - ['X44', 'X38']
   Component 3: 6 features - ['X23', 'X16', 'X8', 'X37', 'X12', 'X3']
   Component 4: 2 features - ['X15', 'X8']
   Component 5: 4 features - ['X49', 'X34', 'X30', 'X29']
   Component 6: 3 features - ['X24', 'X23', 'X10']
   Component 7: 3 features - ['X22', 'X11', 'X6']
   Component 8: 2 features - ['X48', 'X32']
   Component 9: 2 features - ['X41', 'X39']
   Component 10: 2 features - ['X21', 'X16']
   Expected confounders from input: 23 pairs
   Status: ⚠️  Mismatch in confounder count

✓ Check 3: Topological Orderi

## 8. Summary Statistics

Let's create a summary table of the component structure.

In [9]:
# Create summary table
summary_data = []
for comp_idx, component in enumerate(components):
    parent_features = parents_dict.get(comp_idx, [])
    summary_data.append({
        'Component': comp_idx,
        'Size': len(component),
        'Confounded': 'Yes' if confounded_info.get(comp_idx, False) else 'No',
        'Num_Parents': len(parent_features),
        'Is_Source': 'Yes' if len(parent_features) == 0 else 'No',
        'Features': ', '.join([feature_names[i] for i in component[:3]]) + ('...' if len(component) > 3 else '')
    })

summary_df = pd.DataFrame(summary_data)

print("="*70)
print("COMPONENT SUMMARY TABLE")
print("="*70)
print(summary_df.to_string(index=False))

print(f"\n📊 Statistics:")
print(f"   Total components: {len(components)}")
print(f"   Source components (no parents): {summary_df[summary_df['Is_Source'] == 'Yes'].shape[0]}")
print(f"   Confounded components: {summary_df[summary_df['Confounded'] == 'Yes'].shape[0]}")
print(f"   Average component size: {summary_df['Size'].mean():.2f}")
print(f"   Largest component size: {summary_df['Size'].max()}")
print(f"   Smallest component size: {summary_df['Size'].min()}")

COMPONENT SUMMARY TABLE
 Component  Size Confounded  Num_Parents Is_Source         Features
         0     5        Yes            0       Yes  X21, X35, X7...
         1     3        Yes            1        No     X20, X7, X31
         2     2        Yes            0       Yes         X44, X38
         3     6        Yes            1        No  X23, X16, X8...
         4     2        Yes            0       Yes          X15, X8
         5     4        Yes            0       Yes X49, X34, X30...
         6     3        Yes            0       Yes    X24, X23, X10
         7     3        Yes            0       Yes     X22, X11, X6
         8     2        Yes            1        No         X48, X32
         9     2        Yes            1        No         X41, X39
        10     2        Yes            1        No         X21, X16
        11     1         No            0       Yes               X2
        12     1         No            0       Yes              X28
        13     1        

## 9. Test Post-Interventional Sampling (Optional)

Quick test to ensure the sampling mechanism works correctly.

In [10]:
print("="*70)
print("TESTING POST-INTERVENTIONAL SAMPLING")
print("="*70)

# Take a test instance
test_instance = X_test.iloc[0].values

# Test with empty coalition (sample all features)
print(f"\nTest 1: Empty coalition S = [] (sample all features)")
try:
    samples_empty = causal_explainer._sample_post_interventional(S=[], x_instance=test_instance)
    print(f"   ✅ Success! Generated {samples_empty.shape[0]} samples")
    print(f"   Sample shape: {samples_empty.shape}")
    print(f"   Sample range: [{samples_empty.min():.3f}, {samples_empty.max():.3f}]")
except Exception as e:
    print(f"   ❌ Error: {type(e).__name__}: {str(e)}")

# Test with full coalition (fix all features)
print(f"\nTest 2: Full coalition S = all features (fix all)")
try:
    S_full = list(range(len(feature_names)))
    samples_full = causal_explainer._sample_post_interventional(S=S_full, x_instance=test_instance)
    print(f"   ✅ Success! Generated {samples_full.shape[0]} samples")
    print(f"   All samples identical to instance: {np.allclose(samples_full, test_instance)}")
except Exception as e:
    print(f"   ❌ Error: {type(e).__name__}: {str(e)}")

# Test with random coalition
print(f"\nTest 3: Random coalition S = first 10 features")
try:
    S_partial = list(range(10))
    samples_partial = causal_explainer._sample_post_interventional(S=S_partial, x_instance=test_instance)
    print(f"   ✅ Success! Generated {samples_partial.shape[0]} samples")
    print(f"   Fixed features match: {np.allclose(samples_partial[:, S_partial], test_instance[S_partial])}")
    print(f"   Sampled features vary: {np.var(samples_partial[:, 10:]) > 0}")
except Exception as e:
    print(f"   ❌ Error: {type(e).__name__}: {str(e)}")

print(f"\n{'='*70}")
print("✅ CausalShapley initialization and sampling tests complete!")
print(f"{'='*70}")

TESTING POST-INTERVENTIONAL SAMPLING

Test 1: Empty coalition S = [] (sample all features)
   ❌ Error: IndexError: index 50 is out of bounds for axis 0 with size 50

Test 2: Full coalition S = all features (fix all)
   ❌ Error: IndexError: index 50 is out of bounds for axis 0 with size 50

Test 3: Random coalition S = first 10 features
   ❌ Error: IndexError: index 50 is out of bounds for axis 0 with size 50

✅ CausalShapley initialization and sampling tests complete!
